# Pre-trained Models aur Transfer Learning

CNN ko train karne mein bahut time lag sakta hai, aur us kaam ke liye bahut saara data chahiye hota hai. Halaanki, zyada tar time un best low-level filters ko seekhne mein jaata hai jinhe network images se patterns nikaalne ke liye use karta hai. Ek natural sawaal uthta hai - kya hum ek dataset par trained neural network ko use karke, poori training process ke bina, usse alag images classify karne ke liye adapt kar sakte hain?

Is approach ko **Transfer Learning** kehte hain, kyunki hum ek neural network model se doosre mein kuch knowledge transfer karte hain. Transfer learning mein, hum aam taur par ek pre-trained model se shuru karte hain, jise kisi bade image dataset jaise **ImageNet** par train kiya gaya hota hai. Yeh models pehle se hi general images se alag-alag features nikaalne ka achha kaam kar sakte hain, aur kai cases mein un nikaale gaye features ke upar sirf ek classifier banana bhi achhe results de sakta hai.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import os
from tfcv import *

## Cats vs. Dogs Dataset

Is unit mein, hum cats aur dogs ki images classify karne ki ek real-life problem solve karenge. Isi wajah se, hum [Kaggle Cats vs. Dogs Dataset](https://www.kaggle.com/c/dogs-vs-cats) use karenge, jise [Microsoft se bhi download kiya ja sakta hai](https://www.microsoft.com/en-us/download/details.aspx?id=54765&WT.mc_id=academic-77998-cacaste).

Chaliye is dataset ko download karte hain aur `data` directory mein extract karte hain (isme thoda time lag sakta hai!):

In [ ]:
if not os.path.exists('data/kagglecatsanddogs_5340.zip'):
    !wget -P data https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

In [ ]:
import zipfile
if not os.path.exists('data/PetImages'):
    with zipfile.ZipFile('data/kagglecatsanddogs_5340.zip', 'r') as zip_ref:
        zip_ref.extractall('data')

Afsos ki baat hai, dataset mein kuch corrupt image files hain. Humein corrupt files check karne ke liye ek quick cleanup karna hoga. Is tutorial ko clutter se bachane ke liye, humne dataset check karne wala code ek module mein shift kar diya hai.

In [ ]:
check_image_dir('data/PetImages/Cat/*.jpg')
check_image_dir('data/PetImages/Dog/*.jpg')

## Dataset load karna

Pichhle examples mein, hum Keras mein built-in datasets load kar rahe the. Ab hum apne khud ke dataset se deal karne wale hain, jise humein images ki directory se load karna hoga.

Real life mein, image datasets ka size kaafi bada ho sakta hai, aur yeh mumkin nahi hota ki poora data memory mein fit ho jaye. Isliye, datasets ko aksar **generators** ke roop mein represent kiya jaata hai jo training ke liye suitable minibatches mein data return kar sakte hain.

Image classification se deal karne ke liye, Keras ek special function `image_dataset_from_directory` include karta hai, jo alag-alag classes ke corresponding subdirectories se images load kar sakta hai. Yeh function images ko scale karne ka bhi dhyan rakhta hai, aur yeh dataset ko training aur test subsets mein bhi split kar sakta hai:

In [ ]:
data_dir = 'data/PetImages'
batch_size = 64
ds_train = keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split = 0.2,
    subset = 'training',
    seed = 13,
    image_size = (224,224),
    batch_size = batch_size
)
ds_test = keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split = 0.2,
    subset = 'validation',
    seed = 13,
    image_size = (224,224),
    batch_size = batch_size
)

Dono calls ke liye same `seed` value set karna important hai, kyunki yeh train aur test datasets ke beech images ke split ko affect karta hai.

Dataset directories se class names automatically le leta hai, aur zaroorat padne par aap inhe call karke access kar sakte hain:

In [ ]:
ds_train.class_names

Jo datasets humne paaye hain unhe seedhe model ko train karne ke liye `fit` function ko pass kiya ja sakta hai. Inme corresponding images aur labels dono hote hain, jinhe neeche di gayi construction use karke loop kiya ja sakta hai:

In [ ]:
for x,y in ds_train:
    print(f"Training batch shape: features={x.shape}, labels={y.shape}")
    x_sample, y_sample = x,y
    break
    
display_dataset(x_sample.numpy().astype(np.int),np.expand_dims(y_sample,1),classes=ds_train.class_names)

> **Note**: Dataset ki saari images floating point tensors ke roop mein 0-255 range ke saath represent hoti hain. Unhe neural network mein bhejne se pehle, humein un values ko 0-1 range mein scale karna hoga. Images ko plot karte waqt, humein ya toh yahi karna hoga, ya values ko `int` type mein convert karna hoga (jo humne upar ke code mein kiya hai), taaki `matplotlib` ko dikhaya ja sake ki hum original unscaled image plot karna chahte hain.

## Pre-trained Models

Kai image classification tasks ke liye pre-trained neural network models mil sakte hain. In models mein se kai `keras.applications` namespace ke andar available hain, aur internet par aur bhi zyada models mil sakte hain. Chaliye dekhte hain ki sabse simple VGG-16 model ko kaise load aur use kiya ja sakta hai:

In [ ]:
# SHA-256 of VGG16 weights (with top)
VGG16_WEIGHTS_SHA256 = '64373286793e3c8b2b4e3219cbf3544bce2f55ab4682710a29f5e7af8f5e4f61'

_weights_path = keras.utils.get_file(
    'vgg16_weights_tf_dim_ordering_tf_kernels.h5',
    'https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels.h5',
    file_hash=VGG16_WEIGHTS_SHA256,
    hash_algorithm='sha256',
)

vgg = keras.applications.VGG16(weights=_weights_path)
inp = keras.applications.vgg16.preprocess_input(x_sample[:1])

res = vgg(inp)
print(f"Most probable class = {tf.argmax(res,1)}")

keras.applications.vgg16.decode_predictions(res.numpy())

Yahan kuch important baatein hain:
* Kisi bhi pre-trained network ko input dene se pehle use ek khaas tareeke se pre-process karna padta hai. Yeh corresponding `preprocess_input` function ko call karke kiya jaata hai, jo images ka ek batch leta hai, aur unka processed form return karta hai. VGG-16 ke case mein, images ko normalize kiya jaata hai, aur har channel ke liye kuch pre-defined average value ghata di jaati hai. Aisa isliye kiya jaata hai kyunki VGG-16 ko originally isi pre-processing ke saath train kiya gaya tha.
* Neural network input batch par apply hota hai, aur humein result mein 1000-element wale tensor ka ek batch milta hai jo har class ki probability dikhata hai. Is tensor par `argmax` call karke hum sabse probable class number pa sakte hain.
* Jo result milta hai woh [ek `ImageNet` class ka number hai](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a). Is result ko samajhne ke liye, hum `decode_predictions` function bhi use kar sakte hain, jo top n classes ko unke names ke saath return karta hai.

Chaliye VGG-16 network ki architecture bhi dekhte hain:

In [ ]:
vgg.summary()

## GPU computation

Deep neural networks, jaise VGG-16 aur doosri zyada modern architectures, ko run karne ke liye kaafi computational power chahiye hoti hai. Agar available ho toh GPU acceleration use karna samajhdaari hai. Khushkismati se, Keras agar GPU available ho toh computations ko automatically GPU par speed up kar deta hai. Hum neeche diye gaye code se check kar sakte hain ki Tensorflow GPU use kar paa raha hai ya nahi:

In [ ]:
tf.config.list_physical_devices('GPU')

## VGG features extract karna

Agar hum apni images se features nikaalne ke liye VGG-16 use karna chahte hain, toh humein final classification layers ke bina model chahiye. Hum is code se VGG-16 model ko top layers ke bina instantiate kar sakte hain:

In [ ]:
vgg = keras.applications.VGG16(include_top=False)

inp = keras.applications.vgg16.preprocess_input(x_sample[:1])
res = vgg(inp)
print(f"Shape after applying VGG-16: {res[0].shape}")
plt.figure(figsize=(15,3))
plt.imshow(res[0].numpy().reshape(-1,512))

Feature tensor ka dimension 7x7x512 hai, lekin ise dekhne ke liye humein ise 2D form mein reshape karna pada.

Ab dekhte hain ki kya un features ko images classify karne ke liye use kiya ja sakta hai. Chaliye manually kuch images ka hissa lete hain (hamare case mein 50 minibatches), aur unke feature vectors pehle se calculate kar lete hain. Hum yeh karne ke liye Tensorflow **dataset** API use kar sakte hain. `map` function ek dataset leta hai aur use transform karne ke liye ek diya gaya lambda-function apply karta hai. Hum is mechanism ka use naye datasets banane ke liye karte hain, `ds_features_train` aur `ds_features_test`, jinmein original images ke bajaye VGG-extracted features hote hain.

In [ ]:
num = batch_size*50
ds_features_train = ds_train.take(50).map(lambda x,y : (vgg(x),y))
ds_features_test = ds_test.take(10).map(lambda x,y : (vgg(x),y))

for x,y in ds_features_train:
    print(x.shape,y.shape)
    break

Humne performance tez karne ke liye dataset size ko limit karne ke liye `.take(50)` construction use kiya. Aap bilkul is experiment ko poore dataset par kar sakte hain.

Ab jab hamare paas extracted features wala dataset hai, toh hum cat aur dog ke beech farq karne ke liye ek simple dense classifier train kar sakte hain. Yeh network (7,7,512) shape ka feature vector lega, aur ek output dega jo ya toh dog ya cat se belong karega. Kyunki yeh binary classification hai, hum `sigmoid` activation function aur `binary_crossentropy` loss use karte hain.

In [ ]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(7,7,512)),
    keras.layers.Dense(1,activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
hist = model.fit(ds_features_train, validation_data=ds_features_test)

Result shaandaar hai, hum lagbhag 95% probability ke saath ek cat aur ek dog ke beech farq kar sakte hain! Halaanki, humne is approach ko sirf saari images ke ek subset par test kiya hai, kyunki manual feature extraction bahut time lene wala lagta hai.

## VGG network ka use karke Transfer Learning

Hum manually features ko pehle se calculate karne se bhi bach sakte hain, original VGG-16 network ko training ke dauraan poore roop mein use karke, apne network mein pehli layer ke roop mein feature extractor add karke.

Keras architecture ki khoobsurati yeh hai ki upar humne jo VGG-16 model define kiya hai, use kisi doosre neural network mein ek layer ke roop mein bhi use kiya ja sakta hai! Humein bas us par ek dense classifier banana hai, aur phir poore network ko backpropagation use karke train karna hai.

In [ ]:
model = keras.models.Sequential()
model.add(keras.applications.VGG16(include_top=False,input_shape=(224,224,3)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(1,activation='sigmoid'))

model.layers[0].trainable = False

model.summary()

Yeh model ek end-to-end classification network jaisa dikhta hai, jo ek image leta hai aur class return karta hai. Halaanki, yahan tricky baat yeh hai ki hum chahte hain ki VGG16 ek feature extractor ke roop mein kaam kare, aur dobara train na ho. Isliye, humein **convolutional feature extractor ke weights ko freeze karna** hoga. Hum `model.layers[0]` call karke network ki pehli layer tak pahunch sakte hain, aur humein bas `trainable` property ko `False` set karna hai.

> **Note**: Feature extractor ke weights ko freeze karna zaroori hai, kyunki warna untrained classifier layer convolutional extractor ke original pre-trained weights ko bigaad sakti hai.

Aap dekhenge ki hamare network mein total parameters lagbhag 15 million hain, lekin hum sirf 25k parameters train kar rahe hain. Upar ke level ke convolutional filters ke baaki saare parameters pre-trained hain. Yeh achhi baat hai, kyunki hum kam examples ke saath kam parameters ko fine-tune kar paate hain.

Ab hum apna network train karenge aur dekhenge ki hum kitna achha kar sakte hain. Running time relatively lamba ho sakta hai, aur agar execution kuch der ke liye freeze lage toh chinta mat kijiye.

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
hist = model.fit(ds_train, validation_data=ds_test)

Lagta hai humne cats vs. dogs classifier kaafi accuracy ke saath paa liya hai!

## Model save aur load karna

Ek baar jab humne model train kar liya, toh hum model ki architecture aur trained weights ko future use ke liye ek file mein save kar sakte hain:

In [ ]:
model.save('data/cats_dogs.tf')

Hum phir kabhi bhi file se model load kar sakte hain. Yeh tab useful ho sakta hai jab agla experiment model ko bigaad de - tab aapko dobara shuru nahi karna padega.

In [ ]:
model = keras.models.load_model('data/cats_dogs.tf')

## Transfer Learning ko Fine-tune karna

Pichhle section mein, humne final classifier layer ko apne dataset mein images classify karne ke liye train kiya tha. Halaanki, humne feature extractor ko dobara train nahi kiya, aur hamara model un features par depend kar raha tha jo model ne ImageNet data par seekhe the. Agar aapke objects general ImageNet images se visually alag hain, toh features ka yeh combination sabse achha kaam nahi kar sakta. Isliye convolutional layers ko bhi train karna shuru karna samajhdaari hogi.

Aisa karne ke liye, hum un convolutional filter parameters ko unfreeze kar sakte hain jinhe humne pehle freeze kiya hua tha.

> **Note:** Yeh important hai ki aap pehle parameters ko freeze karein aur classification layer mein weights ko stable karne ke liye kai epochs tak training karein. Agar aap turant unfrozen parameters ke saath end-to-end network ki training shuru karte hain, toh bade errors convolutional layers mein pre-trained weights ko bigaad sakte hain.

Hamara convolutional VGG-16 model pehli layer ke andar located hai, aur yeh khud kai layers se bana hai. Hum uske structure par ek nazar daal sakte hain:

In [ ]:
model.layers[0].summary()

Hum convolutional base ki saari layers ko unfreeze kar sakte hain:

In [ ]:
model.layers[0].trainable = True

Halaanki, sabko ek saath unfreeze karna sabse achha idea nahi hai. Hum sabse pehle sirf kuch aakhri convolution layers ko hi unfreeze kar sakte hain, kyunki unmein high-level patterns hote hain jo hamari images ke liye relevant hain. Jaise, shuruaat ke liye, hum sabhi layers ko chhodkar aakhri 4 ko freeze kar sakte hain:

In [ ]:
for i in range(len(model.layers[0].layers)-4):
    model.layers[0].layers[i].trainable = False
model.summary()

Dhyan dijiye ki trainable parameters ki sankhya kaafi badh gayi hai, lekin yeh phir bhi saare parameters ka lagbhag 50% hi hai.

Unfreezing ke baad, hum kuch aur epochs ki training kar sakte hain (hamare example mein, hum sirf ek karenge). Aap pre-trained weights par asar kam karne ke liye kam learning rate bhi chun sakte hain. Halaanki, kam learning rate ke saath bhi, aap training ki shuruaat mein accuracy ghatne ki ummeed kar sakte hain, jab tak woh end mein fixed weights ke case se thoda zyada level tak na pahunch jaye.

> **Note:** Yeh training bahut dheemi hoti hai, kyunki humein network ki kai layers ke zariye gradients ko wapas propagate karna padta hai!

In [ ]:
hist = model.fit(ds_train, validation_data=ds_test)

Hum sambhavtah higher training accuracy paayenge, kyunki hum zyada powerful network use kar rahe hain jisme zyada parameters hain, lekin validation accuracy utni zyada nahi badhegi.

Network ki kuch aur layers ko unfreeze karke aur zyada training dene mein bejhijhak rahiye, yeh dekhne ke liye ki kya aap higher accuracy paa sakte hain!

## Doosre Computer Vision Models

VGG-16 sabse simple computer vision architectures mein se ek hai. Keras kai aur pre-trained networks provide karta hai. Unmein se sabse zyada use hone wale **ResNet** architectures hain, jo Microsoft ne develop kiye hain, aur **Inception** jo Google ne banaya hai. Jaise, chaliye sabse simple ResNet-50 model ki architecture explore karte hain (ResNet alag-alag depth wale models ka ek family hai, agar aap sach mein deep model dekhna chahte hain toh aap ResNet-152 ke saath experiment kar sakte hain):

In [ ]:
resnet = keras.applications.ResNet50()
resnet.summary()

Jaisa ki aap dekh sakte hain, model mein wahi jaane-pehchane building blocks hain: convolutional layers, pooling layers aur final dense classifier. Hum is model ko bilkul usi tareeke se transfer learning ke liye use kar sakte hain jaise hum VGG-16 use kar rahe hain. Aap upar diye gaye code ke saath experiment kar sakte hain, alag-alag ResNet models ko base model ke roop mein use karke, aur dekh sakte hain ki accuracy kaise badalti hai.

## Batch Normalization

Is network mein ek aur type ki layer hoti hai: **Batch Normalization**. Batch Normalization ka idea yeh hai ki neural network se guzarne wali values ko sahi range mein laya jaye. Aam taur par neural networks tab sabse achha kaam karte hain jab saari values [-1,1] ya [0,1] ke range mein hoti hain, aur isiliye hum apne input data ko isi hisaab se scale/normalize karte hain. Halaanki, deep network ki training ke dauraan, aisa ho sakta hai ki values is range se kaafi bahar chali jayein, jisse training mein problem ho jaati hai. Batch Normalization layer current minibatch ki saari values ke liye average aur standard deviation calculate karti hai, aur unhe neural network layer mein bhejne se pehle signal ko normalize karne ke liye use karti hai. Isse deep networks ki stability mein kaafi sudhaar hota hai.

## Key Takeaways

Transfer learning ka use karke, hum apne custom object classification task ke liye jaldi se ek classifier taiyaar kar paaye, aur high accuracy paa sake. Halaanki, yeh example poori tarah fair nahi tha, kyunki original VGG-16 network ko cats aur dogs pehchaanne ke liye pre-train kiya gaya tha, aur isliye hum sirf un patterns ko re-use kar rahe the jo network mein pehle se maujood the. Aap zyada exotic domain-specific objects par, jaise kisi plant mein production line ki details, ya alag-alag tree ke leaves par, kam accuracy ki ummeed kar sakte hain.

Aap dekh sakte hain ki jin zyada complex tasks ko hum ab solve kar rahe hain, unmein higher computational power chahiye hoti hai, aur unhe CPU par aasaani se solve nahi kiya ja sakta. Agle unit mein, hum kam compute resources use karke wahi model train karne ke liye zyada lightweight implementation use karne ki koshish karenge, jisse accuracy thodi kam hogi.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->